
# Stage A — Build Reusable Grid → Polygon Mapping (Simple, Cell-Center)

This notebook creates a **reusable mapping** from ERA5 grid points `(latitude, longitude)` to **basin** and **watershed** polygons using a **cell-center point-in-polygon** approach.

**Why a separate mapping?** Once generated, it can be reused for ERA5, ECMWF/GraphCast forecasts, or any other dataset on the same grid.

**Outputs** (written to `data/spatial/grid_mapping/` by default):
- `era5_grid_to_polygons.parquet`
- `era5_grid_to_polygons.csv`
- `era5_grid_to_polygons_QA.txt` (quick summary)



## 1. Environment & Requirements

> Run this cell once to ensure dependencies are available (uncomment if needed).


In [ ]:

# !pip install pandas geopandas shapely pyproj fiona



## 2. Configuration

Adjust paths only if your repository structure differs.


In [1]:

import os
import glob
import textwrap
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# --- INPUTS ---
# Merged ERA5 grid-level parquet (already produced by your merge notebook; timestamps aren't used here)
ERA5_MERGED_PARQUET = "data/era5/era5_merged/merged_era5_6hour_1979_2025.parquet"

# Polygon directories (basins and watersheds) inside your repository
BASINS_DIR      = "data/boundaries/basins"
WATERSHEDS_DIR  = "data/boundaries/186_watershed"

# --- OUTPUTS ---
OUT_DIR = "data/spatial/grid_mapping"
OUT_BASENAME = "era5_grid_to_polygons"  # -> {OUT_BASENAME}.parquet/.csv and a QA .txt

# Candidate ID columns to auto-detect polygon IDs (fallback creates sequential IDs)
CANDIDATE_ID_COLS = [
    "basin_id","BASIN_ID","Basin_ID","ws_id","WS_ID","watershed_id","Watershed_ID",
    "OBJECTID","FID","ID","id","NAME","Name","name","CODE","code"
]



## 3. Helper Functions


In [ ]:

def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)

def pick_vector_file(dir_path: str):
    """Pick the first vector file (.shp/.gpkg) in a directory (prefers basin/watershed-like names)."""
    candidates = []
    candidates += glob.glob(os.path.join(dir_path, "*.shp"))
    candidates += glob.glob(os.path.join(dir_path, "*.gpkg"))
    if not candidates:
        raise FileNotFoundError(f"No .shp or .gpkg found in {dir_path}")
    preferred = sorted(
        candidates,
        key=lambda p: (0 if any(k in os.path.basename(p).lower() for k in ["basin","watershed","wsh","catch","subbasin"]) else 1, p)
    )
    return preferred[0]

def read_polygons(dir_path: str) -> gpd.GeoDataFrame:
    vec = pick_vector_file(dir_path)
    gdf = gpd.read_file(vec)
    if gdf.crs is None:
        # If CRS is missing, assume WGS84; change if your data differ
        gdf = gdf.set_crs(4326)
    else:
        gdf = gdf.to_crs(4326)
    # choose an ID column or create one
    id_col = None
    for c in CANDIDATE_ID_COLS:
        if c in gdf.columns:
            id_col = c
            break
    if id_col is None:
        id_col = "poly_id"
        gdf[id_col] = range(1, len(gdf)+1)
    return gdf[[id_col, "geometry"]].rename(columns={id_col: "polygon_id"})

def stable_grid_id(df_latlon: pd.DataFrame) -> pd.Series:
    """Stable, collision-resistant ID from (lat, lon)."""
    hashed = pd.util.hash_pandas_object(df_latlon[["latitude","longitude"]], index=False)
    return "g" + hashed.astype("uint64").map(lambda x: format(x, "016x"))

def make_points_gdf(latlon: pd.DataFrame) -> gpd.GeoDataFrame:
    return gpd.GeoDataFrame(
        latlon.copy(),
        geometry=[Point(lon, lat) for lat, lon in zip(latlon["latitude"], latlon["longitude"])],
        crs=4326
    )

def spatial_join_id(points: gpd.GeoDataFrame, polys: gpd.GeoDataFrame, label: str) -> pd.Series:
    """Return Series of polygon IDs for `label` using 'within', then fallback to 'intersects' if needed."""
    joined = gpd.sjoin(points, polys, how="left", predicate="within")
    s = joined["polygon_id"].rename(f"{label}_id")
    if s.isna().mean() > 0.1:  # fallback if many misses
        joined2 = gpd.sjoin(points, polys, how="left", predicate="intersects")
        s2 = joined2["polygon_id"].rename(f"{label}_id")
        s = s.fillna(s2)
    return s



## 4. Load Unique Grid Points from Merged ERA5 Parquet


In [ ]:

# Read only latitude/longitude columns to build the unique grid
if not os.path.exists(ERA5_MERGED_PARQUET):
    raise FileNotFoundError(f"Parquet not found: {ERA5_MERGED_PARQUET}")

df_latlon = pd.read_parquet(ERA5_MERGED_PARQUET, columns=["latitude","longitude"]).dropna(subset=["latitude","longitude"])
grid = df_latlon.drop_duplicates(subset=["latitude","longitude"]).reset_index(drop=True)
grid["grid_id"] = stable_grid_id(grid)
grid = grid[["grid_id","latitude","longitude"]]
print(f"Unique grid cells: {len(grid):,}")
grid.head()



## 5. Load Basin & Watershed Polygons (Standardize CRS to EPSG:4326)


In [ ]:

print(f"Loading basins from: {BASINS_DIR}")
basins = read_polygons(BASINS_DIR)
print(f"Basins loaded: {len(basins):,}")

print(f"Loading watersheds from: {WATERSHEDS_DIR}")
watersheds = read_polygons(WATERSHEDS_DIR)
print(f"Watersheds loaded: {len(watersheds):,}")

basins.head(), watersheds.head()



## 6. Point-in-Polygon Assignment (Cell-Center)
This step assigns each grid cell to a **basin** and a **watershed**.  
If many cells remain unassigned (NaN), we fallback from `within` to `intersects` to account for slivers/precision issues.


In [ ]:

points_gdf = make_points_gdf(grid)

print("Assigning basin IDs…")
basin_ids = spatial_join_id(points_gdf, basins, label="basin")

print("Assigning watershed IDs…")
watershed_ids = spatial_join_id(points_gdf, watersheds, label="watershed")

mapping = points_gdf.drop(columns="geometry").copy()
mapping["basin_id"] = basin_ids.values
mapping["watershed_id"] = watershed_ids.values
mapping["outside_flag"] = mapping["basin_id"].isna() & mapping["watershed_id"].isna()

mapping.head()



## 7. Save Mapping Artifacts


In [ ]:

ensure_dir(OUT_DIR)
out_parquet = os.path.join(OUT_DIR, f"{OUT_BASENAME}.parquet")
out_csv     = os.path.join(OUT_DIR, f"{OUT_BASENAME}.csv")
qa_txt      = os.path.join(OUT_DIR, f"{OUT_BASENAME}_QA.txt")

mapping.to_parquet(out_parquet, index=False)
mapping.to_csv(out_csv, index=False)

outside_n = int(mapping["outside_flag"].sum())
summary = textwrap.dedent(f"""
=== GRID → POLYGON MAPPING (simple, cell-center) ===
Input parquet:  {ERA5_MERGED_PARQUET}
Basins dir:     {BASINS_DIR}
Watersheds dir: {WATERSHEDS_DIR}

Unique grid cells:   {len(mapping):,}
Unmapped (outside):  {outside_n:,}  ({outside_n/len(mapping):.2%})

Example rows:
{mapping.head(5).to_string(index=False)}
""").strip()
print(summary)

with open(qa_txt, "w", encoding="utf-8") as f:
    f.write(summary + "\n")

out_parquet, out_csv, qa_txt



## 8. Quick QA Preview


In [ ]:

mapping.describe(include='all')



## 9. Next Steps

- **Stage B (Aggregation)**: Join this mapping to your merged ERA5 parquet and compute polygon × time (6‑hourly) aggregates for basins and watersheds.
- **Weighted upgrade (later)**: Replace this simple mapping with a fractional overlap weights table and reuse the same Stage B logic.
